In [1]:
import pandas as pd
import numpy as np

dim_household = pd.read_csv(r'D:\Data_Engineering\Housing_Loan_Clone\azure_housing_analytics\dataset\silver\dim_fam.csv')
dim_housing = pd.read_csv(r'D:\Data_Engineering\Housing_Loan_Clone\azure_housing_analytics\dataset\silver\dim_housing.csv')
fact_household = pd.read_csv(r'D:\Data_Engineering\Housing_Loan_Clone\azure_housing_analytics\dataset\gold\fact_household.csv')
dim_lender_terms = pd.read_csv(r'D:\Data_Engineering\Housing_Loan_Clone\azure_housing_analytics\dataset\gold\support\dim_lender_terms.csv')


In [2]:
fact_household.head()

,geographyFk,families,reliability,monthlyIncome,monthlyExpenses,netIncome,capacityToPayPagibig,capacityToPayBank
0,133900000,510,High,40233,32517,7716,14081.55,12069.9
1,137401000,141,Moderate,55502,42349,13153,19425.70,16650.6
2,137405000,29,Very Low,70234,54065,16169,24581.90,21070.2
3,137404000,747,High,40058,32678,7380,14020.30,12017.4
4,137402000,109,Moderate,46379,37595,8784,16232.65,13913.7


In [3]:
dim_lender_terms.head()

,channel,ltv_pct,interest_rate,dti_cap_pct,max_loan_amount,max_term_years,source_confidence
0,Pag-IBIG,0.9,0.0575,0.35,10000000.0,30,official
1,Bank (general),0.8,0.0700,0.30,NaN,20,estimated


In [4]:
terms = dim_lender_terms.set_index("channel").to_dict(orient="index")
pagibig = terms["Pag-IBIG"]
bank = terms["Bank (general)"]

def monthly_pmt(principal_col, annual_rate, years):
    r = annual_rate / 12
    n = years * 12
    pmt = principal_col * r * (1 + r) ** n / ((1 + r) ** n - 1)
    return pmt.round(2)


In [5]:
df = dim_housing.merge(fact_household, on='geographyFk', how='inner')

df["houseMedianPrice"] = df.groupby("geographyFk")["price"].transform("median")
df["listingCount"] = df.groupby("geographyFk")["price"].transform("count")
# pagibig details
df["pagIbigRequiredLoan"] = df["price"] * pagibig["ltv_pct"]
df["pagIbigAmortization"] = monthly_pmt(
    df["pagIbigRequiredLoan"], pagibig["interest_rate"], pagibig["max_term_years"]
)
df["pagIbigCapacity"] = (df["monthlyIncome"] * pagibig["dti_cap_pct"]).round(2)
df["pagIbigGap"] = (df["pagIbigCapacity"] - df["pagIbigAmortization"]).round(2)
df["pagIbigLoanStatus"] = np.where(df["pagIbigGap"] <= 0, "Not Applicable", "Applicable")


# bank details
df["bankRequiredLoan"] = df["price"] * bank["ltv_pct"]
df["bankAmortization"] = monthly_pmt(
    df["bankRequiredLoan"], bank["interest_rate"], bank["max_term_years"]
)
df["bankCapacity"] = (df["monthlyIncome"] * bank["dti_cap_pct"]).round(2)
df["bankGap"] = (df["bankCapacity"] - df["bankAmortization"]).round(2)
df["bankLoanStatus"] = np.where(df["bankGap"] <= 0, "Not Applicable", "Applicable")


df.head()




,id,sourceSlug,sourceName,title,price,priceFormatted,pricePerSqm,floorArea,lotArea,city,...,pagIbigRequiredLoan,pagIbigAmortization,pagIbigCapacity,pagIbigGap,pagIbigLoanStatus,bankRequiredLoan,bankAmortization,bankCapacity,bankGap,bankLoanStatus
0,b3d16489-85e2-4652-9458-469f0c5d700d,metrobank,Metrobank,Townhouse,5769000.0,₱ 5.8M,52445.0,165.00,110.0,mandaue,...,5192100.0,30299.69,12390.35,-17909.34,Not Applicable,4615200.0,35781.60,10620.3,-25161.30,Not Applicable
1,34d3d3c6-c96c-4ea9-ad7c-8c11d0037370,metrobank,Metrobank,With Improvement,7188000.0,₱ 7.2M,59900.0,296.00,120.0,lapu-lapu,...,6469200.0,37752.50,12490.45,-25262.05,Not Applicable,5750400.0,44582.79,10706.1,-33876.69,Not Applicable
2,5e85bcbc-70ae-4747-9287-72d499ae4ebd,metrobank,Metrobank,Condominium,27705000.0,₱ 27.7M,145235.0,190.76,NaN,manila,...,24934500.0,145510.97,14081.55,-131429.42,Not Applicable,22164000.0,171837.26,12069.9,-159767.36,Not Applicable
3,81ff202d-3797-4f3d-ae17-bb38a6587d14,metrobank,Metrobank,With Improvement,7733000.0,₱ 7.7M,27917.0,522.00,277.0,manila,...,6959700.0,40614.92,14081.55,-26533.37,Not Applicable,6186400.0,47963.09,12069.9,-35893.19,Not Applicable
4,34bbb6f2-e009-4bd9-b3dd-2dba18283cd5,metrobank,Metrobank,Vacant Lot,840000.0,₱ 840K,4800.0,NaN,175.0,tarlac,...,756000.0,4411.81,10728.20,6316.39,Applicable,672000.0,5210.01,9195.6,3985.59,Applicable


In [6]:


to_drop_columns = ['id', 'sourceSlug', 'sourceName', 'title', 'price', 'priceFormatted', 'pricePerSqm', 'floorArea', 'lotArea', 'city', 'province', 'isNew', 'daysListed', 'listingScore', 'firstSeenAt', 'families', 'reliability', 'monthlyIncome', 'monthlyExpenses', 'netIncome']

fact_affordability = df.drop(columns=to_drop_columns)
print(fact_affordability.columns.tolist())

# fact_affordability.to_csv(r'D:\Data_Engineering\Housing_Loan_Clone\azure_housing_analytics\dataset\gold\fact_amortize.csv')
print('table saved')

['geographyFk', 'capacityToPayPagibig', 'capacityToPayBank', 'houseMedianPrice', 'listingCount', 'pagIbigRequiredLoan', 'pagIbigAmortization', 'pagIbigCapacity', 'pagIbigGap', 'pagIbigLoanStatus', 'bankRequiredLoan', 'bankAmortization', 'bankCapacity', 'bankGap', 'bankLoanStatus']
table saved


In [7]:
total_listings = len(dim_housing)
matched_listings = len(df)

print(f"Total listings in dim_housing: {total_listings}")
print(f"Listings matched to fact_household: {matched_listings}")
print(f"Unmatched (no household income data for their geography): {total_listings - matched_listings}")
print(f"Coverage: {matched_listings / total_listings:.1%}")

Total listings in dim_housing: 18224
Listings matched to fact_household: 7800
Unmatched (no household income data for their geography): 10424
Coverage: 42.8%


In [14]:
fact_affordability.listingCount

0        46
1       143
2       781
3       781
4       378
       ... 
7795    548
7796    194
7797    194
7798    194
7799    194
Name: listingCount, Length: 7800, dtype: int64